In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Breast Cancer Dataset

Based on Kaggle competition https://www.kaggle.com/competitions/breast-cancer-detection/overview

In [ ]:
X_raw = pd.read_csv('/kaggle/input/breast-cancer-detection/train.csv')

We've loaded the **train** dataset. Let's take a look at our data and see which columns are unnecessary.

In [ ]:
X_raw['diagnosis'].value_counts()

Class imbalance is present in the dataset, but AdaBoost's design allows it to perform well despite this challenge.

In [ ]:
X_raw.describe()

All numerical columns except _**Unnamed: 32**_ have some values, so they can be used. _**Unnamed: 32**_ can be dropped because it contains only **NaN** values and is not even described in the dataset instructions.

In [ ]:
X_raw = X_raw.drop('Unnamed: 32', axis=1)
X_raw = X_raw.drop('id', axis=1)

In [ ]:
X_raw.dtypes

In [ ]:
X_raw['diagnosis'].unique()

Now let's deal with the **diagnosis** column; it contains two unique values (**M** = malignant, **B** = benign). As such data have an order, like *benign* being less dangerous than *malignant*, we can use `OrdinalEncoder`.

In [ ]:
from sklearn.preprocessing import  OrdinalEncoder
enc = OrdinalEncoder()
X_raw['diagnosis_e'] = enc.fit_transform(X_raw[['diagnosis']])
X_raw[['diagnosis', 'diagnosis_e']].head()

Now **B** -> `0` and **M** -> `1`

In [ ]:
corr = X_raw.select_dtypes(include=['float64', 'int64']).corr().round(2)
#px.imshow(corr, text_auto=True).show() #plotly not showing image on Kaggle
#So using seaborn
sns.heatmap(corr,  linewidth=.5)


In [ ]:
X_train = X_raw.drop(['diagnosis', 'diagnosis_e'], axis=1)
y_train = X_raw['diagnosis']

 Let's try **Random Forest** and **AdaBoost**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
clf = RandomForestClassifier(max_depth=2, random_state=42)
cv_results = cross_validate(clf, X_train, y_train, cv=4)
cv_results


**GridSearchCV** helps us find the optimal parameters for our model. However, in some cases, it can cause *overfitting*.

Since the **AdaBoost** classifier gives us better results, we'll use it as our main model.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [   150, 200, 300],
    'learning_rate': [2, 1, 0.5 ],
    'estimator__max_depth': [None, 1,  2, 3]
   # 'base_estimator__min_samples_split': [2, 5, 10]
}
tree_clf = DecisionTreeClassifier(max_depth=2, random_state=42)
ada_clf = AdaBoostClassifier(estimator=tree_clf, n_estimators=50, learning_rate=0.5, random_state=42)
grid_search = GridSearchCV(ada_clf, param_grid, cv=4, scoring='accuracy')
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)
ada_clf = grid_search.best_estimator_
cv_results = cross_validate(ada_clf, X_train, y_train, cv=4)
cv_results

Now let's load **test** dataset for submission and make a prediction for it

In [ ]:
X_test_full = pd.read_csv('/kaggle/input/breast-cancer-detection/test.csv')
X_test = X_test_full.drop('Unnamed: 32', axis=1)
X_test = X_test.drop('id', axis=1)

Since GridSearch was slightly overfitted, we tuned the learning rate and number of estimators, achieving an accuracy of about _**0.97**_ While GridSearch best estimator gives _**0.953**_.

In [ ]:
ada_clf.fit(X_train, y_train)
pred = ada_clf.predict(X_test)
submission = pd.DataFrame({
    'id': X_test_full['id'],
    'diagnosis': pred
})
submission.to_csv('/kaggle/working/submission.csv', index=False)